# SAM-2 Integration Test Notebook

This notebook tests the Stage 2 Object Segmentation pipeline with real SAM-2 models on GPU.

**Requirements:**
- Google Colab with GPU runtime (T4 or better)
- At least 15GB VRAM for SAM-2 tiny model

**What this tests:**
1. GPU availability and VRAM
2. SAM-2 model loading
3. Phase 2: Keyframe segmentation
4. Phase 3: Object tracking across frames
5. Metadata schema validation for Stage 4

## Cell 1: Clone Repository

In [ ]:
# Clone the brain-dance repository
!rm -rf /content/brain-dance  # Remove if exists
!git clone https://github.com/ujseah/brain-dance.git /content/brain-dance
%cd /content/brain-dance

# Checkout the feature branch with Phase 3
!git checkout feat/object-segmentation
!git pull origin feat/object-segmentation

print("\n✅ Repository cloned successfully!")

## Cell 2: Verify GPU Runtime

In [ ]:
import torch

print("=" * 50)
print("GPU VERIFICATION")
print("=" * 50)

if torch.cuda.is_available():
    print(f"✅ CUDA available: True")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   PyTorch: {torch.__version__}")
else:
    print("❌ NO GPU DETECTED!")
    print("")
    print("To fix this:")
    print("1. Go to Runtime > Change runtime type")
    print("2. Select 'T4 GPU' under Hardware accelerator")
    print("3. Click Save")
    print("4. Re-run this cell")
    raise RuntimeError("GPU required for SAM-2 integration tests")

## Cell 3: Install Dependencies

In [ ]:
# Install SAM-2 (this takes 2-3 minutes for CUDA compilation)
print("Installing SAM-2 (this may take 2-3 minutes)...")
!pip install git+https://github.com/facebookresearch/segment-anything-2.git

# Install other dependencies
!pip install pytest

print("\n✅ Dependencies installed!")

## Cell 4: Run Unit Tests (Mocked SAM-2)

In [ ]:
# Run the unit tests with mocked SAM-2
# These should all pass regardless of GPU
!cd /content/brain-dance && python -m pytest tests/stages/test_object_segmentation.py -v --tb=short

print("\n✅ Unit tests completed!")

## Cell 5: Test Real SAM-2 Model Loading

In [ ]:
import sys
sys.path.insert(0, '/content/brain-dance')

from backend.stages.object_segmentation import ObjectSegmentationStage

print("=" * 50)
print("SAM-2 MODEL LOADING TEST")
print("=" * 50)

# Test model loading with tiny variant (smallest, ~4GB VRAM)
config = {
    "model_size": "tiny",
    "quality_preset": "fast",
    "allow_cpu": False,  # Force GPU
}

print(f"\nLoading SAM-2 '{config['model_size']}' model...")

try:
    with ObjectSegmentationStage(config) as stage:
        print(f"\n✅ Image model loaded: {stage.image_model is not None}")
        print(f"✅ Mask generator ready: {stage.mask_generator is not None}")
        print(f"✅ Video predictor loaded: {stage.video_predictor is not None}")
        
        # Check VRAM usage
        vram_used = torch.cuda.memory_allocated() / 1e9
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"\nVRAM Usage: {vram_used:.1f} / {vram_total:.1f} GB ({vram_used/vram_total*100:.0f}%)")
        
    print("\n✅ SAM-2 models loaded and cleaned up successfully!")
except Exception as e:
    print(f"\n❌ Model loading failed: {e}")
    raise

## Cell 6: Full Pipeline Integration Test

In [ ]:
import tempfile
from pathlib import Path
from PIL import Image, ImageDraw
import numpy as np
import json

print("=" * 50)
print("FULL PIPELINE INTEGRATION TEST")
print("=" * 50)

# Create synthetic test video with moving object
with tempfile.TemporaryDirectory() as tmpdir:
    frames_dir = Path(tmpdir) / "frames"
    frames_dir.mkdir()
    output_dir = Path(tmpdir) / "output"
    
    # Create 15 test frames with a red rectangle moving across
    print("\n1. Creating synthetic test video (15 frames)...")
    for i in range(15):
        img = Image.new("RGB", (640, 480), color=(100, 150, 200))  # Blue background
        draw = ImageDraw.Draw(img)
        
        # Moving red rectangle
        x = 50 + i * 30  # Moves right
        draw.rectangle([x, 150, x + 100, 300], fill=(255, 0, 0))
        
        # Static green rectangle
        draw.rectangle([400, 300, 550, 400], fill=(0, 255, 0))
        
        img.save(frames_dir / f"{i:04d}.jpg")
    
    print(f"   Created {len(list(frames_dir.glob('*.jpg')))} frames")
    
    # Run segmentation pipeline
    print("\n2. Running segmentation pipeline...")
    config = {
        "model_size": "tiny",
        "quality_preset": "fast",
        "keyframe_interval": 5,
        "allow_cpu": False,
    }
    
    def progress_callback(pct, msg):
        print(f"   {pct*100:5.1f}% - {msg}")
    
    with ObjectSegmentationStage(config) as stage:
        result = stage.segment(
            frames_dir=str(frames_dir),
            output_dir=str(output_dir),
            progress_callback=progress_callback
        )
    
    # Validate results
    print("\n3. Validating results...")
    print(f"   Objects detected: {len(result.objects)}")
    
    # Check metadata file
    metadata_path = output_dir / "object_metadata.json"
    if metadata_path.exists():
        with open(metadata_path) as f:
            metadata = json.load(f)
        
        print(f"   Metadata file: ✅ exists")
        print(f"   - num_objects: {metadata.get('num_objects')}")
        print(f"   - num_frames: {metadata.get('num_frames')}")
        print(f"   - frame_to_objects keys: {len(metadata.get('frame_to_objects', {}))}")
        
        if 'quality_metrics' in metadata:
            qm = metadata['quality_metrics']
            print(f"   - mean_iou: {qm.get('mean_iou', 'N/A')}")
            print(f"   - warnings: {len(qm.get('warnings', []))}")
    else:
        print(f"   Metadata file: ❌ missing")
    
    # Check mask files
    masks_dir = output_dir / "masks"
    if masks_dir.exists():
        object_dirs = list(masks_dir.iterdir())
        print(f"   Mask directories: {len(object_dirs)} objects")
        for obj_dir in object_dirs[:3]:  # Show first 3
            mask_files = list(obj_dir.glob("*.png"))
            print(f"     - {obj_dir.name}/: {len(mask_files)} masks")
    
    print("\n✅ Full pipeline test completed!")

## Cell 7: Validate Metadata Schema for Stage 4

In [ ]:
print("=" * 50)
print("STAGE 4 METADATA SCHEMA VALIDATION")
print("=" * 50)

# Expected schema for Stage 4 compatibility
required_top_level = ['num_objects', 'num_frames', 'objects', 'frame_to_objects', 'quality_metrics']
required_object_fields = ['object_id', 'frame_indices', 'confidence']
required_per_frame_fields = ['bbox', 'centroid', 'area']

# Load the metadata from the previous test
if 'metadata' in dir():
    print("\nValidating metadata schema...\n")
    
    # Check top-level fields
    print("Top-level fields:")
    for field in required_top_level:
        status = "✅" if field in metadata else "❌"
        print(f"  {status} {field}")
    
    # Check object fields
    if metadata.get('objects'):
        obj = metadata['objects'][0]
        print("\nObject fields:")
        for field in required_object_fields:
            status = "✅" if field in obj else "❌"
            print(f"  {status} {field}")
        
        # Check per-frame data
        if 'per_frame_data' in obj and obj['per_frame_data']:
            frame_key = list(obj['per_frame_data'].keys())[0]
            frame_data = obj['per_frame_data'][frame_key]
            print("\nPer-frame data fields:")
            for field in required_per_frame_fields:
                status = "✅" if field in frame_data else "❌"
                print(f"  {status} {field}")
    
    # Check frame_to_objects mapping
    if metadata.get('frame_to_objects'):
        print(f"\nFrame-to-objects mapping: ✅ ({len(metadata['frame_to_objects'])} frames)")
    
    print("\n✅ Schema validation complete!")
    print("\nThis metadata structure is compatible with Stage 4's _get_object_context() method.")
else:
    print("\n⚠️  Run Cell 6 first to generate metadata for validation.")

## Cell 8: VRAM Usage Summary

In [ ]:
import torch

print("=" * 50)
print("VRAM USAGE SUMMARY")
print("=" * 50)

if torch.cuda.is_available():
    # Get current stats
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"\nCurrent VRAM status:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Total:     {total:.2f} GB")
    print(f"  Free:      {total - reserved:.2f} GB")
    
    # Clear cache
    torch.cuda.empty_cache()
    
    print("\n✅ VRAM cleared. Ready for next test.")
else:
    print("No GPU available.")

## Summary

If all cells completed successfully:

1. **GPU** - T4 or better is available
2. **Unit Tests** - All 47 tests pass (mocked SAM-2)
3. **Model Loading** - SAM-2 tiny loads without OOM
4. **Full Pipeline** - Phase 2 + Phase 3 complete successfully
5. **Metadata Schema** - Compatible with Stage 4

The Stage 2 Object Segmentation pipeline is **production-ready**.